In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [2]:
nav = pd.read_csv("../data/processed/clean_nav.csv")
txn = pd.read_csv("../data/processed/clean_transactions.csv")
portfolio = pd.read_csv("../data/processed/clean_portfolio_holdings.csv")
fund = pd.read_csv("../data/processed/clean_fund_master.csv")
sharpe_df = pd.read_csv("../data/processed/sharpe_values.csv")

In [3]:
nav['date'] = pd.to_datetime(nav['date'])

In [5]:
txn['transaction_date'] = pd.to_datetime(txn['transaction_date'])

In [6]:
nav = nav.sort_values(['amfi_code', 'date'])
nav['daily_return'] = nav.groupby('amfi_code')['nav'].pct_change()
nav_returns = nav.dropna(subset=['daily_return']).copy()

In [7]:
var_cvar_list = []

for code, group in nav_returns.groupby('amfi_code'):
    returns = group['daily_return']
    
    # Historical VaR at 95%
    var_95 = np.percentile(returns, 5)
    
    # CVaR = mean of returns below VaR
    cvar_95 = returns[returns <= var_95].mean()
    
    var_cvar_list.append({
        'amfi_code': code,
        'var_95': round(var_95 * 100, 4),
        'cvar_95': round(cvar_95 * 100, 4)
    })

var_cvar_df = pd.DataFrame(var_cvar_list)
var_cvar_df = var_cvar_df.merge(
    fund[['amfi_code', 'scheme_name', 'fund_house']], 
    on='amfi_code', how='left')

print(var_cvar_df.sort_values('var_95').head(10))

    amfi_code  var_95  cvar_95  \
22     119599 -2.6859  -3.2384   
17     119095 -2.6188  -3.1667   
4      101207 -2.6021  -3.2459   
11     118634 -2.5438  -3.2304   
21     119598 -2.4507  -3.0595   
39     149324 -2.3483  -3.1036   
7      102886 -1.9220  -2.3251   
2      100033 -1.9034  -2.3456   
25     120505 -1.8892  -2.4342   
16     119094 -1.8480  -2.4260   

                                          scheme_name  \
22          SBI Small Cap Fund - Direct Plan - Growth   
17             Axis Small Cap Fund - Regular - Growth   
4              ABSL Small Cap Fund - Regular - Growth   
11     Nippon India Small Cap Fund - Regular - Growth   
21         SBI Small Cap Fund - Regular Plan - Growth   
39              DSP Small Cap Fund - Regular - Growth   
7                 UTI Mid Cap Fund - Regular - Growth   
2   HDFC Mid-Cap Opportunities Fund - Regular - Gr...   
25           ICICI Pru Midcap Fund - Regular - Growth   
16                Axis Midcap Fund - Regular - Growth  

In [8]:
var_cvar_df.to_csv("../data/processed/var_cvar_report.csv", index=False)